In [6]:
# ============================================================
# Aggregate option counts by autonomy+pair across the dataset
# Pair 01: EPI01, MOR01, REL01 (dominated role) – counts A/B/C totals across ALL domains
# Pair 02: EPI02, MOR02, REL02 (dominant role)  – counts A/B/C totals across ALL domains
# Inputs: CSVs with columns: scenario_code, answer
# Outputs: per-model CSVs + combined CSVs; inline prints
# ============================================================

import os
import re
import pandas as pd

# ---- Config ----
OUTPUT_DIR = "./outputs_autonomy_pairs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = [
    "aya-expanse-32b_test.csv",
    "gemma3-12b_test.csv",
    "gemma3-27b_test.csv",
    "gpt4o_test.csv",
    "llama3.1_test.csv",
    "llama3.2-latest_test.csv",
    "llama3.3-latest_test.csv",
    "mistral-small-3.2-24b_test.csv",
    "mistral-small3.1-latest_test.csv",
    "qwen2.5-14b_test.csv",
    "qwen2.5-32b_test.csv",
    "qwen2.5-72b_test.csv"
]

# ---- Helpers ----
def extract_letter(val: str):
    """Extract A/B/C from the answer; return None if not found."""
    if pd.isna(val):
        return None
    s = str(val).strip().upper()
    for pat in [
        r'^\s*([ABC])\s*$',            # "A"
        r'\bOPTION\s*([ABC])\b',       # "Option C"
        r'\bANSWER[:\s]*([ABC])\b',    # "Answer: B"
        r'(^|\s)[\(\[]?([ABC])[\)\].:]\s',  # "(C) ..." or "B) ..."
        r'\b([ABC])\b',                # standalone letter
        r'\b([ABC])\s*$'               # ends with letter
    ]:
        m = re.search(pat, s)
        if m:
            # some patterns capture in group 2
            return (m.group(1) if m.group(1) in ['A','B','C']
                    else (m.group(2) if m.lastindex and m.group(2) in ['A','B','C'] else None))
    return None

def parse_code(code: str):
    """'EDU01_EPI01' -> (autonomy='EPI', pair_no=1). Domain is irrelevant for this analysis."""
    if not isinstance(code, str):
        return None, None
    s = code.strip().upper()
    m = re.match(r'^[A-Z0-9]+_(EPI|MOR|REL)(\d{2})$', s)
    if not m:
        return None, None
    autonomy, pair = m.groups()
    return autonomy, int(pair)

def ensure_abc_cols(pivot: pd.DataFrame):
    for col in ['A','B','C']:
        if col not in pivot.columns:
            pivot[col] = 0
    return pivot[['A','B','C']]

def summarize_by_pair(df_parsed: pd.DataFrame, pair_no: int) -> pd.DataFrame:
    """
    From parsed df with columns [autonomy, pair_no, answer_letter],
    return counts for the requested pair_no aggregated ACROSS ALL DOMAINS:
      rows = autonomy in [EPI,MOR,REL]
      cols = A/B/C + total + majority
    """
    sub = df_parsed[df_parsed['pair_no'] == pair_no]
    if sub.empty:
        # return zeros for EPI/MOR/REL to keep shape
        base = pd.DataFrame({'autonomy':['EPI','MOR','REL']})
        base['A'] = 0; base['B'] = 0; base['C'] = 0
        base['total'] = 0; base['majority_option'] = ''
        base['majority_count'] = 0
        return base

    counts = (sub.groupby(['autonomy','answer_letter'])
                  .size().reset_index(name='count'))
    pivot = counts.pivot(index='autonomy', columns='answer_letter', values='count').fillna(0).astype(int)
    pivot = ensure_abc_cols(pivot)
    pivot['total'] = pivot[['A','B','C']].sum(axis=1)
    pivot['majority_option'] = pivot[['A','B','C']].idxmax(axis=1)
    pivot['majority_count'] = pivot[['A','B','C']].max(axis=1)
    # order autonomy
    pivot = pivot.reindex(['EPI','MOR','REL'])
    return pivot.reset_index()

def show(df: pd.DataFrame, title: str):
    print(f"\n=== {title} ===")
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string(index=False))

# ---- Process models ----
all_pair01 = []
all_pair02 = []

for path in csv_files:
    if not os.path.exists(path):
        print(f"[WARN] Missing file: {path} — skipped.")
        continue

    model = os.path.basename(path).replace("_test.csv","")
    df = pd.read_csv(path)

    # locate columns
    cols = {c.lower(): c for c in df.columns}
    sc_col = cols.get('scenario_code') or next((c for c in df.columns if 'scenario' in c.lower()), None)
    ans_col = cols.get('answer') or next((c for c in df.columns if 'answer' in c.lower()), None)
    if sc_col is None or ans_col is None:
        print(f"[WARN] {model}: missing scenario_code/answer columns.")
        continue

    # parse
    tmp = df[[sc_col, ans_col]].rename(columns={sc_col:'scenario_code', ans_col:'answer'})
    tmp['answer_letter'] = tmp['answer'].apply(extract_letter)
    parsed = tmp['scenario_code'].apply(parse_code).apply(pd.Series)
    parsed.columns = ['autonomy','pair_no']
    tmp = pd.concat([tmp, parsed], axis=1)
    tmp = tmp.dropna(subset=['answer_letter','autonomy','pair_no']).copy()
    tmp['pair_no'] = tmp['pair_no'].astype(int)

    # Summaries across ALL DOMAINS by autonomy+pair
    pair01 = summarize_by_pair(tmp, pair_no=1)
    pair02 = summarize_by_pair(tmp, pair_no=2)
    pair01.insert(0, 'model', model)
    pair02.insert(0, 'model', model)

    # Save per model
    out01 = os.path.join(OUTPUT_DIR, f"{model}_pair01_autonomy_counts.csv")
    out02 = os.path.join(OUTPUT_DIR, f"{model}_pair02_autonomy_counts.csv")
    pair01.to_csv(out01, index=False)
    pair02.to_csv(out02, index=False)

    # Inline view
    show(pair01, title=f"{model} — Pair 01 (EPI01/MOR01/REL01) counts across ALL domains")
    show(pair02, title=f"{model} — Pair 02 (EPI02/MOR02/REL02) counts across ALL domains")

    print(f"[OK] {model}")
    print(f"     Pair 01 CSV → {out01}")
    print(f"     Pair 02 CSV → {out02}")

    all_pair01.append(pair01)
    all_pair02.append(pair02)

# ---- Combined across models ----
if all_pair01:
    comb01 = pd.concat(all_pair01, ignore_index=True)
    comb01_out = os.path.join(OUTPUT_DIR, "ALL_models_pair01_autonomy_counts.csv")
    comb01.to_csv(comb01_out, index=False)
    show(comb01, title="ALL MODELS — Pair 01 counts (by autonomy, across ALL domains)")
    print(f"[OK] Combined Pair 01 → {comb01_out}")

if all_pair02:
    comb02 = pd.concat(all_pair02, ignore_index=True)
    comb02_out = os.path.join(OUTPUT_DIR, "ALL_models_pair02_autonomy_counts.csv")
    comb02.to_csv(comb02_out, index=False)
    show(comb02, title="ALL MODELS — Pair 02 counts (by autonomy, across ALL domains)")
    print(f"[OK] Combined Pair 02 → {comb02_out}")



=== aya-expanse-32b — Pair 01 (EPI01/MOR01/REL01) counts across ALL domains ===


answer_letter,model,autonomy,A,B,C,total,majority_option,majority_count
0,aya-expanse-32b,EPI,5,6,9,20,C,9
1,aya-expanse-32b,MOR,4,3,13,20,C,13
2,aya-expanse-32b,REL,5,3,12,20,C,12



=== aya-expanse-32b — Pair 02 (EPI02/MOR02/REL02) counts across ALL domains ===


answer_letter,model,autonomy,A,B,C,total,majority_option,majority_count
0,aya-expanse-32b,EPI,3,7,9,19,C,9
1,aya-expanse-32b,MOR,1,5,14,20,C,14
2,aya-expanse-32b,REL,3,4,13,20,C,13


[OK] aya-expanse-32b
     Pair 01 CSV → ./outputs_autonomy_pairs\aya-expanse-32b_pair01_autonomy_counts.csv
     Pair 02 CSV → ./outputs_autonomy_pairs\aya-expanse-32b_pair02_autonomy_counts.csv

=== gemma3-12b — Pair 01 (EPI01/MOR01/REL01) counts across ALL domains ===


...

[OK] qwen2.5-72b
     Pair 01 CSV → ./outputs_autonomy_pairs\qwen2.5-72b_pair01_autonomy_counts.csv
     Pair 02 CSV → ./outputs_autonomy_pairs\qwen2.5-72b_pair02_autonomy_counts.csv

=== ALL MODELS — Pair 01 counts (by autonomy, across ALL domains) ===


answer_letter,model,autonomy,A,B,C,total,majority_option,majority_count
0,aya-expanse-32b,EPI,5,6,9,20,C,9
1,aya-expanse-32b,MOR,4,3,13,20,C,13
2,aya-expanse-32b,REL,5,3,12,20,C,12
3,gemma3-12b,EPI,2,3,15,20,C,15
4,gemma3-12b,MOR,4,3,13,20,C,13
5,gemma3-12b,REL,3,7,10,20,C,10
6,gemma3-27b,EPI,5,2,13,20,C,13
7,gemma3-27b,MOR,3,3,14,20,C,14
8,gemma3-27b,REL,3,5,12,20,C,12
9,gpt4o,EPI,4,3,13,20,C,13


[OK] Combined Pair 01 → ./outputs_autonomy_pairs\ALL_models_pair01_autonomy_counts.csv

=== ALL MODELS — Pair 02 counts (by autonomy, across ALL domains) ===


answer_letter,model,autonomy,A,B,C,total,majority_option,majority_count
0,aya-expanse-32b,EPI,3,7,9,19,C,9
1,aya-expanse-32b,MOR,1,5,14,20,C,14
2,aya-expanse-32b,REL,3,4,13,20,C,13
3,gemma3-12b,EPI,1,3,16,20,C,16
4,gemma3-12b,MOR,1,5,14,20,C,14
5,gemma3-12b,REL,0,4,16,20,C,16
6,gemma3-27b,EPI,2,2,16,20,C,16
7,gemma3-27b,MOR,1,5,14,20,C,14
8,gemma3-27b,REL,0,5,15,20,C,15
9,gpt4o,EPI,1,6,13,20,C,13


[OK] Combined Pair 02 → ./outputs_autonomy_pairs\ALL_models_pair02_autonomy_counts.csv
